In [ ]:
import sys
import os
sys.path.append('../')
import pathlib as pl
from SymEigen import *
from sympy import symbols
from project_dir import backend_source_dir

Gen = EigenFunctionGenerator()
Gen.MacroBeforeFunction("__host__ __device__")


In [ ]:
D, dHat, kappa, xi = symbols('D dHat kappa xi')
Cl = Gen.Closure(kappa, D, dHat, xi)
# classic log barrier with thickness (codim-shell form)
kB = - kappa * (D - xi * xi - 2 * xi * dHat - dHat * dHat) **2 * log((D-xi * xi)/(2 * xi * dHat + dHat * dHat))
# Stiff-GIPC-style stiff log^2 barrier for zero-thickness (volumetric) contacts
Cl2 = Gen.Closure(kappa, D, dHat)
kB2 = kappa * (D - dHat * dHat) ** 2 * log(D / (dHat * dHat)) ** 2
kB, kB2


In [ ]:
dkBdd = kB.diff(D)
dkB2dd = kB2.diff(D)
dkBdd, dkB2dd


In [ ]:
ddkBddd = dkBdd.diff(D)
ddkB2ddd = dkB2dd.diff(D)
ddkBddd, ddkB2ddd


In [ ]:
s = f'''
// > Squared Version
// > D := d*d

{Cl("KappaBarrierWithThickness",kB)}
{Cl("dKappaBarrierWithThicknessdD",dkBdd)}
{Cl("ddKappaBarrierWithThicknessddD",ddkBddd)}
{Cl2("KappaBarrierLog2",kB2)}
{Cl2("dKappaBarrierLog2dD",dkB2dd)}
{Cl2("ddKappaBarrierLog2ddD",ddkB2ddd)}
'''

# Hand-written dispatcher: zero-thickness contacts (volumetric IPC) use the stiff
# log^2 barrier (Stiff-GIPC design); contacts with thickness (codim shells) keep the
# classic log barrier. Public names (KappaBarrier etc.) are kept so all callers
# (simplex PT/EE/PE/PP, vertex-half-plane ground, friction normal_force) follow.
dispatcher = '''
/* Dispatcher: xi == 0 -> stiff log^2 barrier (volumetric, Stiff-GIPC design);
 * xi > 0  -> classic log barrier with thickness (codim shells). */
template <typename T>
__host__ __device__ void KappaBarrier(T& R, const T& kappa, const T& D, const T& dHat, const T& xi)
{
    if(xi == 0.0)
        KappaBarrierLog2(R, kappa, D, dHat);
    else
        KappaBarrierWithThickness(R, kappa, D, dHat, xi);
}
template <typename T>
__host__ __device__ void dKappaBarrierdD(T& R, const T& kappa, const T& D, const T& dHat, const T& xi)
{
    if(xi == 0.0)
        dKappaBarrierLog2dD(R, kappa, D, dHat);
    else
        dKappaBarrierWithThicknessdD(R, kappa, D, dHat, xi);
}
template <typename T>
__host__ __device__ void ddKappaBarrierddD(T& R, const T& kappa, const T& D, const T& dHat, const T& xi)
{
    if(xi == 0.0)
        ddKappaBarrierLog2ddD(R, kappa, D, dHat);
    else
        ddKappaBarrierWithThicknessddD(R, kappa, D, dHat, xi);
}
'''
print(s + dispatcher)

f = open( backend_source_dir('cuda') / 'contact_system/contact_models/sym/codim_ipc_contact.inl', 'w')
f.write(s + dispatcher)
f.close()
